# 🚗 Treinamento YOLO11m para Detecção de Tráfego
**Google Colab - GPU Gratuita**

Este notebook treina um modelo YOLO11m para detecção de veículos em vídeos de tráfego.

## Como usar:
1. Ative a GPU: **Menu → Runtime → Change runtime type → GPU → Save**
2. Execute cada célula em ordem
3. Baixe o modelo treinado ao final

**Tempo estimado:** 30-60 minutos para 50 épocas com GPU.
Sem GPU: pode levar várias horas.


In [ ]:
# ============================================================================
# PASSO 1: Instalar dependências
# ============================================================================
!pip install -q ultralytics opencv-python numpy
print("✅ Dependências instaladas")

In [ ]:
# ============================================================================
# PASSO 2: Baixar dataset do GitHub (ou fazer upload manual)
# ============================================================================
import os
import subprocess

# Tentar baixar do GitHub releases
dataset_url = 'https://github.com/LanzaDev/traffic_analysis/releases/download/v1.0/dataset.zip'
print('📥 Baixando dataset do GitHub...')
!wget -q '{dataset_url}' -O dataset.zip || echo 'Download falhou - será necessário fazer upload manual'

import os
if os.path.exists('dataset.zip'):
    print('✅ Dataset baixado com sucesso!')
    !unzip -q dataset.zip
    print('✅ Dataset extraído')
else:
    print('⚠️ Download falhou. Faça upload manual das pastas: assets/train, assets/val, labels/train, labels/val')
    print('   Ou use o arquivo de exemplo fornecido no repositório')

In [ ]:
# ============================================================================
# PASSO 3: Criar arquivo data.yaml
# ============================================================================
import os

yaml_content = '''train: dataset/train/images
val: dataset/val/images
nc: 6
names:
  0: person
  1: car
  2: motorcycle
  3: bus
  4: truck
  5: articulated_truck
'

with open('data.yaml', 'w') as f:
    f.write(yaml_content)

print('✅ Arquivo data.yaml criado')
print(open('data.yaml').read())

In [ ]:
# ============================================================================
# PASSO 5: Treinar o modelo
# ============================================================================
import torch
from ultralytics import YOLO

device = '0' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('⚠️ AVISO: GPU não detectada. Treinando em CPU (será mais lento).')
else:
    print(f'✅ GPU detectada: {{torch.cuda.get_device_name(0)}}')

model = YOLO('yolo11m.pt')

results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project='runs/train',
    name='traffic_analysis',
    exist_ok=True,
    patience=20,
    save=True,
    verbose=True
)

print('\n✅ Treinamento concluído!')
print(f"📁 Modelo salvo em: runs/train/traffic_analysis/weights/best.pt")

In [ ]:
# ============================================================================
# PASSO 6: Validar o modelo treinado
# ============================================================================
from ultralytics import YOLO

model = YOLO('runs/train/traffic_analysis/weights/best.pt')

metrics = model.val(
    data='data.yaml',
    imgsz=640,
    batch=16,
    device=device
)

print('\n✅ Validação concluída!')
print(f"   mAP50: {{metrics.box.map50:.4f}}")
print(f"   mAP50-95: {{metrics.box.map:.4f}}")
print(f"   Precisão: {{metrics.box.mp:.4f}}")
print(f"   Recall: {{metrics.box.mr:.4f}}")

In [ ]:
# ============================================================================
# PASSO 7: Baixar o modelo treinado
# ============================================================================
from google.colab import files

model_path = 'runs/train/traffic_analysis/weights/best.pt'

import os
if os.path.exists(model_path):
    print(f"📥 Baixando modelo: {{model_path}}")
    files.download(model_path)
    print('✅ Download iniciado!')
else:
    print('❌ Modelo não encontrado. Verifique se o treinamento foi concluído.')